# App data export
- Small files for the store manager app

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, numpy as np, json

DRIVE = Path('/content/drive/MyDrive/m5_project')
OUT   = DRIVE / 'outputs'
APP   = DRIVE / 'app_data'
APP.mkdir(exist_ok=True)
VAL_DAYS = list(range(1886, 1914))

long = pd.read_parquet(DRIVE / 'processed' / 'ca12_long.parquet',
                       columns=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'd', 'date', 'sales'])
for c in ['id', 'item_id', 'dept_id', 'cat_id', 'store_id']:
    long[c] = long[c].astype(str)
dates = long.drop_duplicates('d').set_index('d')['date']

# 1. item list (one row per product in each store)
items = long.drop_duplicates('id')[['id', 'item_id', 'dept_id', 'cat_id', 'store_id']].set_index('id')

# 2. H-Pro forecasts for the 28 test days, long format
fc = pd.read_parquet(OUT / 'final_forecast_H-Pro-Top_seed0.parquet')
fc.columns = fc.columns.astype(int)
fc = fc.rename_axis('id').reset_index().melt(id_vars='id', var_name='d', value_name='forecast')
fc['d'] = fc['d'].astype(int)
fc['date'] = fc['d'].map(dates)
fc['forecast'] = fc['forecast'].clip(lower=0).round(2)

# 3. real sales: validation month + test month (for "forecast vs actual" charts)
sales = long[long.d >= VAL_DAYS[0]][['id', 'd', 'date', 'sales']]

# 4. daily forecast error per item, from the SAME chosen trials on the validation month
ts   = pd.read_csv(OUT / 'student_trial_scores_seed0.csv')
best = {s: int(ts[ts.set == s].set_index('trial')['H-Pro-Top'].idxmin()) for s in ['store', 'dept']}
val  = sum(pd.read_parquet(OUT / 'student_trials_seed0' / f'{s}_{t:02d}_val.parquet') for s, t in best.items()) / 2
val.columns = val.columns.astype(int)
actual_val = (long[long.d.isin(VAL_DAYS)].pivot(index='id', columns='d', values='sales')
                  .reindex(index=val.index, columns=VAL_DAYS).fillna(0))
items['sigma'] = np.sqrt(((actual_val - val) ** 2).mean(axis=1)).round(3)

# 5. demo stock on hand (M5 has no inventory data): 3-21 days of cover, fixed seed
avg = fc.groupby('id')['forecast'].mean()
rng = np.random.default_rng(0)
items['on_hand'] = (avg.reindex(items.index) * rng.uniform(3, 21, len(items))).round().astype(int)

# 6. model info for the About page
info = {'method': 'H-Pro-Top (store + dept ensemble)', 'chosen_trials': best,
        'test_R_H': round(float(pd.read_csv(OUT / 'student_results_seed0.csv', index_col=0).loc['H-Pro-Top', 'R_H']), 3),
        'forecast_start': str(dates[1914].date()), 'forecast_end': str(dates[1941].date()),
        'stock_note': 'on_hand is demo data (M5 has no inventory)'}

items.reset_index().to_parquet(APP / 'items.parquet', index=False)
fc.to_parquet(APP / 'forecasts.parquet', index=False)
sales.to_parquet(APP / 'sales.parquet', index=False)
(APP / 'model_info.json').write_text(json.dumps(info, indent=2))

for f in sorted(APP.iterdir()):
    print(f'{f.name:<22} {f.stat().st_size / 1e6:.2f} MB')
print(info)
assert len(items) == 6098 and len(fc) == 6098 * 28

Mounted at /content/drive
forecasts.parquet      0.56 MB
items.parquet          0.09 MB
model_info.json        0.00 MB
sales.parquet          0.28 MB
{'method': 'H-Pro-Top (store + dept ensemble)', 'chosen_trials': {'store': 6, 'dept': 6}, 'test_R_H': 0.745, 'forecast_start': '2016-04-25', 'forecast_end': '2016-05-22', 'stock_note': 'on_hand is demo data (M5 has no inventory)'}
